# Aralin 13 - Memorya ng Ahente


## Setup

Ipinapakita ng notebook na ito kung paano gumawa ng isang travel booking agent na may **persistent memory** gamit ang **Microsoft Agent Framework** (MAF).

Matututuhan mo kung paano nakakaapekto ang iba't ibang uri ng agent memory — working, short-term, at long-term — sa kung paano nagtatabi at gumagamit ang agent ng impormasyon sa mga pag-uusap.

**Mga Kinakailangan:**
- Isang Azure AI Foundry project na may na-deploy na chat model (halimbawa `gpt-4o-mini`).
- Nakalog-in gamit ang Azure CLI — patakbuhin ang `az login` sa iyong terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — ang endpoint ng iyong Azure AI Foundry project.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ang pangalan ng iyong na-deploy na modelo.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Mga Uri ng Memorya ng Ahente

Maaaring gamitin ng mga AI na ahente ang iba't ibang uri ng memorya, bawat isa ay may kanya-kanyang layunin:

### Working Memory
Ang mismong thread ng usapan — ang mga mensaheng ipinagpalitan sa isang sesyon. Maaaring balikan ng ahente ang mga naunang mensahe sa parehong thread upang mapanatili ang pagkakaugnay-ugnay. Sa MAF, ito ay nililikha sa pamamagitan ng **`agent.create_session()`**, na nagbabalik ng isang `AgentSession`.

### Short-Term Memory
Impormasyon na nananatili habang tumatagal ang isang gawain o sesyon ngunit hindi permanenteng iniimbak. Halimbawa, maaaring makalikom ang ahente ng mga katotohanan sa isang multi-turn na pag-uusap tungkol sa pagpaplano at gamitin ito upang makabuo ng pinal na itinerary.

### Long-Term Memory
Mga kagustuhan at katotohanan na nananatili **sa pagitan ng mga sesyon**. Hindi na kailangang ulitin ng bumabalik na gumagamit ang kanilang mga dietary restrictions o istilo ng paglalakbay. Kadalasang sinusuportahan ng isang panlabas na imbakan — database, file, o vector index — ang long-term memory at ipinapakita sa ahente gamit ang mga tool.


## Working Memory with Sessions

Ang pinakasimpleng anyo ng memorya ay ang session ng pag-uusap. Kapag ipinasa mo ang parehong session (na nilikha sa pamamagitan ng `agent.create_session()`) sa mga sunud-sunod na `agent.run()` na tawag, nakikita ng agent ang buong kasaysayan ng pag-uusap na iyon at maaaring maalala ang mga naunang detalye.

Gumawa tayo ng isang travel agent at ipakita ang working memory.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Tamang na naaalala ng ahente ang budget dahil pareho ang session ng dalawang mensahe. Ito ay **working memory** — umiiral lamang ito habang buhay ang session.

### Ano ang nangyayari sa isang bagong thread?

Kung gagawa tayo ng **bagong** session, wala nang alaala ang ahente tungkol sa naunang usapan:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pattern ng Pangmatagalang Memorya

Upang matandaan ang mga kagustuhan ng gumagamit **sa iba't ibang sesyon**, kailangan namin ng isang permanenteng imbakan na nasa labas ng usapan. Nilalapitan ng ahente ang imbakan na ito sa pamamagitan ng **mga tool** — mga function na maaari nitong tawagan upang i-save at kunin ang impormasyon.

Nasa ibaba ang isang simpleng in-memory preference store (sa produksyon, susuportahan ito ng database o vector index) at ilalantad ito bilang mga tool na maaaring gamitin ng ahente.

### Arkitektura  
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Bagong user na nagbu-book ng anniversary trip

Bumista si Sarah sa unang pagkakataon. Dapat itala ng ahente ang kanyang mga kagustuhan gamit ang mga tools at gamitin ito para magrekomenda ng mga hotel.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Bumababalik si Sarah makalipas ang ilang linggo

Nagsisimula si Sarah ng **ganap na bagong thread** (pinapalabas ang isang bagong session). Walang laman ang working memory, ngunit nandiyan pa rin ang kanyang impormasyon sa long-term preference store. Dapat kuhanin ito ng agent at gamitin upang gawing personal ang mga rekomendasyon.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

Sa araling ito natutunan mo ang tatlong uri ng memorya ng ahente at paano ito ipinatutupad gamit ang Microsoft Agent Framework:

| Memory Type | MAF Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` | Isang pag-uusap lamang |
| **Short-term** | Naipon na konteksto sa loob ng thread | Isang gawain / sesyon |
| **Long-term** | Panlabas na tindahan na naa-access gamit ang `@tool` functions | Sa pagitan ng mga sesyon |

### Key Takeaways
1. **`agent.create_session()`** ay nagbibigay ng working memory — nakikita ng ahente ang buong kasaysayan ng pag-uusap sa loob ng isang sesyon.
2. **Ang mga bagong sesyon ay nawawala ang konteksto** — kung walang long-term memory, hindi matandaan ng ahente ang mga nakaraang pag-uusap.
3. **Ang `@tool` functions** ay nagsisilbing tulay — pinapayagan nila ang ahente na mag-imbak at kumuha ng impormasyon mula sa isang permanenteng tindahan.
4. **Ang personalisasyon ay bumubuti sa paglipas ng panahon** — kung mas marami ang naka-imbak na kagustuhan, mas maganda ang mga rekomendasyon ng ahente.

### Real-World Applications
- **Customer Service**: Tandaan ang kasaysayan at mga kagustuhan ng customer
- **Personal Assistants**: Panatilihin ang konteksto sa loob ng mga araw o linggo
- **Healthcare**: Subaybayan ang impormasyon at mga kagustuhan ng pasyente
- **E-commerce**: Personal na pamimili base sa kasaysayan

### Next Steps
- Palitan ang in-memory dict ng isang database o vector store (hal. Azure AI Search)
- Magdagdag ng memory expiration para sa mga impormasyong sensitibo sa oras
- Bumuo ng multi-agent system na may shared memory
- Tuklasin ang Cognee notebook para sa knowledge-graph-backed memory


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay isinalin gamit ang serbisyong AI na pagsasalin na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong mga salin ay maaaring maglaman ng mga pagkakamali o hindi wastong pagsasalin. Ang orihinal na dokumento sa wikang nilikha nito ang dapat ituring na opisyal na pinagmulan. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
